# STEP 4 — Axis 3: Paraphrase Consistency

**What this step does:**
- For each question, runs the model on the original + 5 paraphrases (6 total)
- Both `direct` and `cot` modes, all 7 models
- Measures: % agreement across paraphrases per question
- Computes `consistency_delta = consistency(cot) - consistency(direct)` per model

**Outputs produced:**
- `results/consistency_raw.csv` — per-question-per-paraphrase predictions
- `results/consistency_summary.csv` — mean consistency per (model, mode, dataset)
- `results/consistency_delta.csv` — delta (CoT - direct) per (model, dataset)

**Estimated run time:** ~8–10 hours (this is the heaviest step: 400 questions × 6 variants × 7 models × 2 modes)

**Tip:** Run this as a long Kaggle session. It checkpoints after every model so you can resume if interrupted.

**Prerequisite:** Steps 1 must be complete (`data/paraphrases.json` must exist)

In [ ]:
import sys, json, gc
import torch
import pandas as pd
from pathlib import Path

sys.path.insert(0, '/kaggle/working/light_paper')

BASE_DIR    = Path('/kaggle/working/light_paper')
DATA_DIR    = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'

from src.models    import MODEL_REGISTRY, load_model
from src.prompts   import make_consistency_prompt
from src.inference import generate_responses, parse_gsm8k_answer, parse_truthfulqa_answer
from src.metrics   import compute_consistency_by_model, compute_consistency_delta

with open(DATA_DIR / 'datasets.json') as f:
    all_data = json.load(f)

with open(DATA_DIR / 'paraphrases.json') as f:
    paraphrases = json.load(f)

CON_DATASETS = ['gsm8k', 'truthfulqa']
MODES        = ['direct', 'cot']
PARSERS      = {'gsm8k': parse_gsm8k_answer, 'truthfulqa': parse_truthfulqa_answer}

print('Ready.')

In [ ]:
# ── Build expanded question list: original + 5 paraphrases ────────────────
def expand_with_paraphrases(samples: list, paraphrases: dict, ds_name: str) -> list:
    """
    For each sample, return one entry per paraphrase (idx 0 = original question).
    Returns list of dicts: {question_id, paraphrase_idx, question, answer, dataset}
    """
    expanded = []
    for s in samples:
        qid = s['id']
        paras = paraphrases.get(qid, [])
        variants = [s['question']] + paras  # idx 0 = original
        for idx, q in enumerate(variants[:6]):  # cap at 6 (original + 5)
            expanded.append({
                'question_id':    qid,
                'paraphrase_idx': idx,
                'question':       q,
                'answer':         s['answer'],
                'dataset':        ds_name,
            })
    return expanded

In [ ]:
# ── Resume-safe ────────────────────────────────────────────────────────────
RAW_PATH = RESULTS_DIR / 'consistency_raw.csv'

if RAW_PATH.exists():
    existing_df = pd.read_csv(RAW_PATH)
    done_keys = set(zip(existing_df['model'], existing_df['mode'], existing_df['dataset']))
    print(f'⚡ Resuming — {len(done_keys)} combos done')
else:
    existing_df = pd.DataFrame()
    done_keys = set()

all_rows = existing_df.to_dict('records') if not existing_df.empty else []

In [ ]:
# ── Main consistency loop ─────────────────────────────────────────────────
for model_config in MODEL_REGISTRY:
    for mode in MODES:
        pending = [
            ds for ds in CON_DATASETS
            if (model_config.name, mode, ds) not in done_keys
        ]
        if not pending:
            print(f'✓ Skip {model_config.name}/{mode}')
            continue

        print(f'\n── {model_config.name} | {mode} ──────────────────')
        model, tokenizer = load_model(model_config)

        for ds_name in pending:
            samples  = all_data[ds_name]
            parser   = PARSERS[ds_name]
            expanded = expand_with_paraphrases(samples, paraphrases, ds_name)

            prompts = [
                make_consistency_prompt(e['question'], mode, ds_name)
                for e in expanded
            ]
            print(f'  [{ds_name}] {len(prompts)} prompts ({len(samples)} questions × ~6 variants)...')

            responses = generate_responses(
                model, tokenizer, prompts,
                max_new_tokens=512, batch_size=4
            )

            for e, resp in zip(expanded, responses):
                predicted = parser(resp) or ''
                all_rows.append({
                    'model':          model_config.name,
                    'size_b':         model_config.size_b,
                    'mode':           mode,
                    'dataset':        ds_name,
                    'question_id':    e['question_id'],
                    'paraphrase_idx': e['paraphrase_idx'],
                    'answer':         e['answer'],
                    'predicted':      predicted,
                })

            # Checkpoint
            pd.DataFrame(all_rows).to_csv(RAW_PATH, index=False)
            done_keys.add((model_config.name, mode, ds_name))

            # Quick consistency preview
            temp_df = pd.DataFrame(all_rows)
            subset = temp_df[
                (temp_df.model == model_config.name) &
                (temp_df.mode == mode) &
                (temp_df.dataset == ds_name)
            ]
            from src.metrics import compute_consistency
            per_q = subset.groupby('question_id')['predicted'].apply(list)
            mean_con = sum(compute_consistency(a) for a in per_q) / len(per_q)
            print(f'  [{ds_name}] mean_consistency = {mean_con:.3f}')

        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        print(f'  GPU freed')

In [ ]:
# ── Compute summaries ─────────────────────────────────────────────────────
df = pd.read_csv(RAW_PATH)

summary = compute_consistency_by_model(df)
summary.to_csv(RESULTS_DIR / 'consistency_summary.csv', index=False)

delta = compute_consistency_delta(summary)
delta.to_csv(RESULTS_DIR / 'consistency_delta.csv', index=False)

print('Consistency by (model, mode, dataset):')
pivot = summary.pivot_table(index='model', columns=['mode','dataset'], values='consistency').round(3)
print(pivot.to_string())

print('\nConsistency Delta (CoT - Direct):')
print(delta[['model','dataset','delta']].to_string(index=False))

print(f'\n✅ STEP 4 COMPLETE')
print(f'   Next: Run step5_analysis_and_plots.ipynb')